In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# load the dataset directly like member 3 did
data = pd.read_csv("../dataset/creditcard.csv")
print(data.shape)

In [ ]:
data.head()

In [ ]:
data.info()
data["Class"].value_counts()

In [ ]:
# separating features and target
X = data.drop('Class', axis=1)
y = data['Class']

# standard scaling amount and time
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

In [ ]:
# split into train and test before applying smote
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('train size:', X_train.shape)
print('test size:', X_test.shape)

In [ ]:
# fix the class imbalance using smote only on training data
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('After SMOTE, class counts:')
print(pd.Series(y_train_resampled).value_counts())

In [ ]:
# train the logistic regression model on the balanced data
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train_resampled, y_train_resampled)
print('training done.')

In [ ]:
# evaluate the model on the unseen test set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
# show the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()